In [1]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# =========================
# 1. CẤU HÌNH FILE
# =========================
INPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/07_esco_it_occupation_skill_relations.csv"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/12_skill_master.xlsx"


# =========================
# 2. HÀM ĐỌC FILE
# =========================
def read_table(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng file chưa hỗ trợ: {path.suffix}")


# =========================
# 3. CHUẨN HÓA TEXT
# =========================
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    text = re.sub(r"[^a-z0-9+#./\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# =========================
# 4. HÀM TÌM CỘT
# =========================
def find_column(df: pd.DataFrame, candidates: list[str], required=True):
    cols_map = {normalize_text(c): c for c in df.columns}

    for cand in candidates:
        cand_norm = normalize_text(cand)
        if cand_norm in cols_map:
            return cols_map[cand_norm]

    for col in df.columns:
        col_norm = normalize_text(col)
        for cand in candidates:
            cand_norm = normalize_text(cand)
            if cand_norm in col_norm or col_norm in cand_norm:
                return col

    if required:
        raise KeyError(
            f"Không tìm thấy cột phù hợp. Candidates={candidates}. "
            f"Các cột hiện có: {list(df.columns)}"
        )
    return None


# =========================
# 5. CHẠY CHÍNH
# =========================
def main():
    df = read_table(INPUT_FILE)
    print("Đã đọc file:", df.shape)
    print("Các cột hiện có:", df.columns.tolist())

    # Tìm cột skill
    skill_id_col = find_column(
        df,
        ["skill_id", "id_skill", "skillId", "skill_id_uri", "skill_uri", "conceptUri"],
        required=False
    )

    skill_name_col = find_column(
        df,
        ["skill_name", "preferredLabel_skill", "skillLabel", "skill_label", "preferredLabel", "label", "name"]
    )

    skill_type_col = find_column(
        df,
        ["skill_type", "type", "relation_type"],
        required=False
    )

    group_col = find_column(
        df,
        ["group"],
        required=False
    )

    # Rename cho dễ xử lý
    rename_map = {
        skill_name_col: "skill_name"
    }
    if skill_id_col:
        rename_map[skill_id_col] = "skill_id"
    if skill_type_col:
        rename_map[skill_type_col] = "skill_type"
    if group_col:
        rename_map[group_col] = "group"

    df_work = df.rename(columns=rename_map).copy()

    # Nếu thiếu cột thì thêm
    if "skill_id" not in df_work.columns:
        df_work["skill_id"] = None
    if "skill_type" not in df_work.columns:
        df_work["skill_type"] = None
    if "group" not in df_work.columns:
        df_work["group"] = None

    # Chuẩn hóa tên skill để loại trùng
    df_work["skill_name_key"] = df_work["skill_name"].apply(normalize_text)

    # Bỏ dòng skill trống
    df_work = df_work[df_work["skill_name_key"] != ""].copy()

    # Loại trùng
    # ưu tiên theo skill_id nếu có, không thì theo tên skill đã chuẩn hóa
    if df_work["skill_id"].notna().sum() > 0:
        df_master = (
            df_work.sort_values(by=["skill_name"])
            .drop_duplicates(subset=["skill_id"], keep="first")
            .copy()
        )
    else:
        df_master = (
            df_work.sort_values(by=["skill_name"])
            .drop_duplicates(subset=["skill_name_key"], keep="first")
            .copy()
        )

    # Đếm mỗi skill xuất hiện bao nhiêu lần trong relation file
    skill_freq = (
        df_work.groupby("skill_name_key")
        .size()
        .reset_index(name="relation_count")
    )

    df_master = df_master.merge(skill_freq, on="skill_name_key", how="left")

    # Sắp xếp đẹp
    df_master = df_master.sort_values(
        by=["skill_name"],
        ascending=True
    ).reset_index(drop=True)

    # Thêm các cột để chuẩn bị cho bước 13 skill mapping
    df_master["skill_group"] = None
    df_master["skill_subgroup"] = None
    df_master["mapped_taxonomy_group"] = None
    df_master["mapped_taxonomy_subgroup"] = None
    df_master["notes"] = None

    # Chọn cột đầu ra
    output_cols = [
        "skill_id",
        "skill_name",
        "skill_type",
        "group",
        "relation_count",
        "skill_group",
        "skill_subgroup",
        "mapped_taxonomy_group",
        "mapped_taxonomy_subgroup",
        "notes"
    ]

    # Chỉ giữ các cột có tồn tại
    output_cols = [c for c in output_cols if c in df_master.columns]
    df_final = df_master[output_cols]

    # Xuất file
    output_path = Path(OUTPUT_FILE)
    if output_path.suffix.lower() in [".xlsx", ".xls"]:
        df_final.to_excel(output_path, index=False)
    else:
        df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Đã tạo file: {OUTPUT_FILE}")
    print("Số skill unique:", len(df_final))
    print(df_final.head(10))


if __name__ == "__main__":
    main()

Đã đọc file: (3275, 9)
Các cột hiện có: ['occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel', 'occupationLabel_clean', 'skillLabel_clean', 'group']
Đã tạo file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/12_skill_master.xlsx
Số skill unique: 1172
  skill_id                            skill_name skill_type     group  \
0     None                           3D lighting  essential      core   
1     None                          3D modelling   optional      core   
2     None                   3D printing process  essential      core   
3     None                          3D texturing  essential      core   
4     None                                  ABAP   optional      core   
5     None                                  AJAX   optional      core   
6     None                                   APL   optional      core   
7     None                               ASP.NET

### Kết quả bước tạo skill master
Từ file quan hệ occupation-skill, đã trích xuất được danh sách skill duy nhất để tạo file `12_skill_master.xlsx`. File này gồm 

1.172 skill unique và các cột nền để phục vụ bước skill mapping tiếp theo, bao gồm nhóm skill, nhóm skill con và nhóm nghề được 

ánh xạ. Đây là dữ liệu đầu vào để xây dựng lớp kết nối giữa skill trong CV/job description và occupation taxonomy.